# Option VIX, CBOE method, all A-share equity-option underlyings - 2026-09-27

**Question:** what is the 30-day implied volatility index of every listed equity-option underlying (SSE and SZSE ETF options, CFFEX index options), what is the most recent value for the **300** family (SSE `510300.SH`, SZSE `159919.SZ`, CFFEX CSI 300 index options), why did it drop in mid-September, and does the number pass independent checks?

All calculation comes from the package `quant_cn.calculation` (plan 03; tests in `tests/calculation/`). This notebook only loads data through `LakeQuery`, calls the package, checks and plots.

Method (CBOE white paper, `plans/architecture/CALCULATION_DESIGN.md` §2): `TermSelector` (two nearest expiries with ≥ 7 days) → `RiskFreeCurve` (SHIBOR, continuous) → `ForwardPriceEstimator` (F, K0 from put-call parity) → `VarianceStripCalculator` (OTM strip, two-zeros cut-off) → `TermInterpolator` (30-day constant maturity) on an `ExpiryClock`:
- **calendar** clock = the white paper (minutes, 365-day year) - the published convention;
- **trading** clock = exchange sessions (21-session target, 244-session year): holidays inside a term add no time.

Data deviations (flagged in `quality_flag`): no bid/ask in `opt_daily`, so prices are closes (settle when close is 0/missing); expiry and observation at the 15:00 close. Data 2026-08-03 → 2026-09-24 (latest session; 09-25 is the Mid-Autumn holiday).

In [1]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from quant_cn.calculation.black76_model import Black76Model
from quant_cn.calculation.expiry_clock import ExpiryClock
from quant_cn.calculation.forward_price_estimator import ForwardPriceEstimator
from quant_cn.calculation.option_chain_builder import OptionChainBuilder
from quant_cn.calculation.risk_free_curve import RiskFreeCurve
from quant_cn.calculation.term_interpolator import TermInterpolator
from quant_cn.calculation.term_selector import TermSelector
from quant_cn.calculation.variance_strip_calculator import VarianceStripCalculator
from quant_cn.calculation.vix_calculator import VixCalculator
from quant_cn.core.config import Config
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.visualization.chart_theme import ChartTheme

START, END = "20260801", "20260924"
FAMILY_300 = ["OP510300.SH", "OP159919.SZ", "OP000300.SH"]
LABELS = {"OP510300.SH": "SSE 300ETF", "OP159919.SZ": "SZSE 300ETF", "OP000300.SH": "CSI 300 index (CFFEX)"}

config = Config.load()
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)

## 1. Inputs from the lake

In [2]:
basic = query.sql("SELECT * FROM curated.opt_basic WHERE exchange IN ('SSE', 'SZSE', 'CFFEX')")
daily = query.sql("SELECT * FROM curated.opt_daily WHERE exchange IN ('SSE', 'SZSE', 'CFFEX') AND trade_date BETWEEN ? AND ?", [START, END])
shibor = query.sql("SELECT * FROM curated.shibor ORDER BY date")
sessions = query.sql("SELECT cal_date FROM curated.trade_cal WHERE is_open = 1 AND cal_date > ? ORDER BY 1", [START])["cal_date"].tolist()
spot = query.sql(
    "SELECT ts_code, trade_date, close FROM curated.fund_daily UNION ALL SELECT ts_code, trade_date, close FROM curated.index_daily"
).pivot_table(index="trade_date", columns="ts_code", values="close").sort_index()

builder = OptionChainBuilder(exclude_adjusted=True)
opt_codes = sorted(basic["opt_code"].unique())
chains = {code: builder.build_chain(basic, daily, code) for code in opt_codes}
pd.DataFrame({code: {"rows": len(c), "days": c["trade_date"].nunique(), "expiries": c["maturity_date"].nunique()}
              for code, c in chains.items()}).T

,rows,days,expiries
OP000016.SH,6516,39,8
OP000300.SH,9168,39,8
OP000852.SH,11498,39,8
OP159901.SZ,4522,39,6
OP159915.SZ,5468,39,6
OP159919.SZ,3760,39,6
OP159922.SZ,4654,39,6
OP510050.SH,3634,39,6
OP510300.SH,3950,39,6
OP510500.SH,3906,39,6


## 2. Compute: every underlying, both clocks

In [3]:
def build_calculator(clock: ExpiryClock) -> VixCalculator:
    return VixCalculator(RiskFreeCurve(shibor), builder, TermSelector(7), ForwardPriceEstimator(),
                         VarianceStripCalculator("price_or_volume"), TermInterpolator(), clock)


calendar_calc = build_calculator(ExpiryClock("calendar"))
trading_calc = build_calculator(ExpiryClock("trading", sessions))
vix = pd.concat([calendar_calc.compute(chain, code) for code, chain in chains.items()], ignore_index=True)
vix_trading = pd.concat([trading_calc.compute(chain, code) for code, chain in chains.items()], ignore_index=True)
vix["vix_trading"] = vix_trading["vix"].values
vix["spot_code"] = vix["underlying"].str.removeprefix("OP")
print(f"{vix['vix'].notna().sum()}/{len(vix)} underlying-days computed; failed rows: {(vix['quality_flag'].str.startswith('failed')).sum()}")
vix.groupby("underlying")[["vix", "vix_trading"]].agg(["mean", "min", "max"]).round(2)

468/468 underlying-days computed; failed rows: 0


vix               vix_trading              
              mean    min    max        mean    min    max
underlying                                                
OP000016.SH  16.39  14.43  20.38       16.61  15.26  19.26
OP000300.SH  18.60  16.52  23.51       18.81  16.38  22.28
OP000852.SH  28.41  23.21  34.74       28.70  25.08  32.93
OP159901.SZ  27.21  21.62  33.53       27.41  24.89  32.16
OP159915.SZ  35.23  27.09  46.24       35.49  30.73  44.43
OP159919.SZ  18.95  15.51  23.67       19.15  17.87  22.66
OP159922.SZ  26.87  21.74  32.98       27.06  24.52  31.70
OP510050.SH  16.11  13.05  20.54       16.29  15.00  19.62
OP510300.SH  18.55  15.18  23.38       18.76  17.38  22.36
OP510500.SH  27.70  22.41  34.72       27.94  24.78  33.33
OP588000.SH  42.76  32.67  61.07       43.11  37.94  58.51
OP588080.SH  42.93  32.66  61.74       43.32  38.20  59.19

## 3. Most recent value (2026-09-24) - the 300 family first

In [4]:
latest = vix[vix["trade_date"] == vix["trade_date"].max()].assign(family_300=lambda d: d["underlying"].isin(FAMILY_300))
latest.sort_values(["family_300", "vix"], ascending=[False, False])[
    ["trade_date", "underlying", "vix", "vix_trading", "days_near", "days_next", "f_near", "k0_near",
     "n_options_near", "n_options_next", "quality_flag"]
].round(3)

,trade_date,underlying,vix,vix_trading,days_near,days_next,f_near,k0_near,n_options_near,n_options_next,quality_flag
77,20260924,OP000300.SH,17.280,18.806,22.0,57.0,4432.385,4400.00,24.0,22.0,ok
233,20260924,OP159919.SZ,17.045,18.747,34.0,62.0,4.707,4.70,11.0,9.0,ok
350,20260924,OP510300.SH,16.963,18.530,34.0,62.0,4.509,4.50,11.0,9.0,ok
467,20260924,OP588080.SH,36.394,39.119,34.0,62.0,1.661,1.65,13.0,9.0,ok
428,20260924,OP588000.SH,36.289,38.806,34.0,62.0,1.711,1.70,13.0,9.0,ok
194,20260924,OP159915.SZ,29.446,32.373,34.0,62.0,3.309,3.30,11.0,9.0,ok
389,20260924,OP510500.SH,24.247,26.734,34.0,62.0,7.610,7.50,11.0,9.0,ok
116,20260924,OP000852.SH,24.132,26.407,22.0,57.0,7538.233,7500.00,27.0,21.0,ok
155,20260924,OP159901.SZ,23.897,26.344,34.0,62.0,3.409,3.40,11.0,9.0,used_settle
272,20260924,OP159922.SZ,22.684,24.741,34.0,62.0,3.092,3.00,10.0,9.0,ok


## 4. Why the 300 VIX dropped in mid-September

On 2026-09-17 the ETF near term rolled from the 09-23 expiry to 10-28. That term spans 41 calendar days but only 23 sessions: the Mid-Autumn (09-25) and National Day (10-01 → 10-07) holidays sit inside it. The white-paper calendar clock spreads the same total variance over those non-trading days, so the index falls; on the trading clock it does not. CFFEX options (expiry 10-16) show the same, smaller effect.

In [5]:
roll = vix[vix["underlying"].isin(FAMILY_300) & vix["trade_date"].between("20260911", "20260924")]
roll.pivot_table(index="trade_date", columns="underlying", values=["vix", "vix_trading", "days_near"]).round(2)

days_near                                 vix              \
underlying OP000300.SH OP159919.SZ OP510300.SH OP000300.SH OP159919.SZ   
trade_date                                                               
20260911           7.0        12.0        12.0       17.94       18.15   
20260914          32.0         9.0         9.0       18.25       18.20   
20260915          31.0         8.0         8.0       18.46       18.11   
20260916          30.0         7.0         7.0       18.86       18.78   
20260917          29.0        41.0        41.0       17.50       16.95   
20260918          28.0        40.0        40.0       16.62       15.51   
20260921          25.0        37.0        37.0       16.65       15.66   
20260922          24.0        36.0        36.0       16.52       15.59   
20260923          23.0        35.0        35.0       16.72       15.99   
20260924          22.0        34.0        34.0       17.28       17.05   

                       vix_trading                          
underlying OP510300.SH OP000300.SH OP159919.SZ OP510300.SH  
trade_date                                                  
20260911         17.84       19.69       19.15       18.78  
20260914         18.00       19.50       18.95       18.72  
20260915         17.92       19.80       19.06       18.85  
20260916         18.61       20.31       20.05       19.84  
20260917         16.29       19.27       19.26       18.78  
20260918         15.18       18.64       18.35       18.08  
20260921         15.31       18.07       17.87       17.58  
20260922         15.22       18.03       17.94       17.60  
20260923         15.54       18.30       18.46       18.10  
20260924         16.96       18.81       18.75       18.53

## 5. Check 1 - known answer

A chain priced with `Black76Model` at a flat 20% volatility must return ≈ 20 on the calendar clock (the same case is a unit test, TC-VC-001).

In [6]:
model = Black76Model()
synthetic = pd.DataFrame([
    {"trade_date": "20260924", "opt_code": "SYN", "ts_code": f"{exp}{side}{k:.2f}", "call_put": side, "strike": round(k, 2),
     "maturity_date": exp, "days": days, "close": model.compute_price(3.0, k, days / 365, 0.014, 0.20, side),
     "settle": np.nan, "vol": 1.0, "oi": 1.0, "adjusted": False}
    for exp, days in (("20261023", 29), ("20261127", 64)) for k in np.arange(1.5, 5.0001, 0.05) for side in ("C", "P")
])
synthetic_vix = build_calculator(ExpiryClock("calendar")).compute(synthetic, "SYN")["vix"].iloc[0]
synthetic_vix

np.float64(20.140729465669533)

## 6. Check 2 - forward vs the underlying close

Put-call parity gives the market's forward; it must sit next to spot. Index options (and CSI 500 / STAR ETF options) trade at the futures-basis discount, so their forward can sit ~1% below spot.

In [7]:
vix["spot"] = [spot.at[d, u] if (d in spot.index and u in spot.columns) else np.nan for d, u in zip(vix["trade_date"], vix["spot_code"], strict=True)]
vix["f_over_spot"] = vix["f_near"] / vix["spot"]
vix.groupby("underlying")["f_over_spot"].agg(["min", "max"]).round(4)

,min,max
underlying,,
OP000016.SH,0.9887,0.9998
OP000300.SH,0.9893,0.9989
OP000852.SH,0.9835,0.9992
OP159901.SZ,0.9895,0.9992
OP159915.SZ,0.9956,1.0006
OP159919.SZ,0.9936,1.0002
OP159922.SZ,0.9888,1.0010
OP510050.SH,0.9954,1.0007
OP510300.SH,0.9939,1.0005


## 7. Check 3 - VIX vs at-the-money implied volatility

`Black76Model.compute_implied_vol` at K0 (average of call and put) for both terms, blended to 30 days with `TermInterpolator` on the calendar clock. VIX integrates the whole smile, so it should sit near and usually slightly above ATM IV.

In [8]:
selector, interpolator = TermSelector(7), TermInterpolator()
atm = []
for row in vix.itertuples():
    if pd.isna(row.vix):
        atm.append(np.nan)
        continue
    day = chains[row.underlying][chains[row.underlying]["trade_date"] == row.trade_date]
    variances = []
    for label, expiry in zip(("near", "next"), selector.list_terms(day), strict=True):
        quotes, _ = builder.build_quotes(day[day["maturity_date"] == expiry])
        k0, f, t, r = getattr(row, f"k0_{label}"), getattr(row, f"f_{label}"), getattr(row, f"t_{label}"), getattr(row, f"rate_{label}")
        ivs = [model.compute_implied_vol(float(quotes.at[k0, ("price", s)]), f, k0, t, r, s) for s in ("C", "P")]
        variances.append(float(np.mean(ivs)) ** 2)
    atm.append(interpolator.compute(variances[0], row.days_near * 1440, variances[1], row.days_next * 1440, 30 * 1440, 365 * 1440))
vix["atm_iv_30d"] = atm
vix["vix_minus_atm"] = vix["vix"] - vix["atm_iv_30d"]
vix.groupby("underlying")["vix_minus_atm"].agg(["min", "mean", "max"]).round(2)

,min,mean,max
underlying,,,
OP000016.SH,1.07,1.45,1.78
OP000300.SH,0.54,0.99,1.50
OP000852.SH,0.57,1.44,2.31
OP159901.SZ,-0.84,1.10,5.34
OP159915.SZ,-1.35,0.60,2.13
OP159919.SZ,-0.07,1.48,2.25
OP159922.SZ,-2.02,0.32,1.93
OP510050.SH,0.44,1.38,2.13
OP510300.SH,0.36,1.23,1.92


## 8. Check 4 - VIX vs realized volatility

20-session close-to-close realized vol of the underlying (annualised with 244 sessions). Implied usually runs above realized; September was very calm (realized ~9-10%), so VIX ≈ 17-18 is roughly twice realized, a normal risk premium rather than a low reading.

In [9]:
realized = np.log(spot).diff().rolling(20).std() * math.sqrt(244) * 100
vix["realized_20d"] = [realized.at[d, u] if (d in realized.index and u in realized.columns) else np.nan for d, u in zip(vix["trade_date"], vix["spot_code"], strict=True)]
vix.groupby("underlying")[["vix", "vix_trading", "realized_20d"]].mean().round(2)

,vix,vix_trading,realized_20d
underlying,,,
OP000016.SH,16.39,16.61,13.79
OP000300.SH,18.60,18.81,17.96
OP000852.SH,28.41,28.70,31.06
OP159901.SZ,27.21,27.41,27.92
OP159915.SZ,35.23,35.49,40.02
OP159919.SZ,18.95,19.15,17.36
OP159922.SZ,26.87,27.06,28.95
OP510050.SH,16.11,16.29,13.68
OP510300.SH,18.55,18.76,17.22


## 9. The 300 family: calendar clock (published convention) vs trading clock

In [10]:
theme = ChartTheme.from_mode("light")
colors = {"OP510300.SH": "#2a78d6", "OP159919.SZ": "#eb6834", "OP000300.SH": "#1baf7a"}
fig = go.Figure()
last = latest.set_index("underlying")
shift = {code: 14 * (1 - rank) for rank, code in enumerate(last.loc[FAMILY_300, "vix"].sort_values(ascending=False).index)}
for code in FAMILY_300:
    s = vix[vix["underlying"] == code].sort_values("trade_date")
    x = pd.to_datetime(s["trade_date"]).dt.strftime("%Y-%m-%d")
    fig.add_trace(go.Scatter(x=x, y=s["vix"], name=f"{LABELS[code]} (calendar)", mode="lines+markers",
                             line={"color": colors[code], "width": 2}, marker={"size": 8},
                             hovertemplate=LABELS[code] + " calendar: %{y:.2f}<extra></extra>"))
    fig.add_trace(go.Scatter(x=x, y=s["vix_trading"], name=f"{LABELS[code]} (trading)", mode="lines",
                             line={"color": colors[code], "width": 2, "dash": "dot"},
                             hovertemplate=LABELS[code] + " trading: %{y:.2f}<extra></extra>"))
    fig.add_annotation(x=x.iloc[-1], y=s["vix"].iloc[-1], text=f"{LABELS[code]} {s['vix'].iloc[-1]:.1f}", showarrow=False,
                       xanchor="left", xshift=8, yshift=shift[code], font={"color": theme.text_secondary, "size": 11})
fig.add_vline(x="2026-09-17", line={"color": theme.text_secondary, "width": 1, "dash": "dash"})
fig.add_annotation(x="2026-09-17", y=1, yref="paper", text="near term rolls over holidays", showarrow=False,
                   xanchor="right", xshift=-4, font={"color": theme.text_secondary, "size": 11})
fig.update_layout(title="CSI 300 option VIX, 2026-08-03 to 2026-09-24 (solid: calendar clock, dotted: trading clock)",
                  hovermode="x unified", paper_bgcolor=theme.surface, plot_bgcolor=theme.surface,
                  font={"color": theme.text_primary}, width=1040, height=500,
                  margin={"l": 60, "r": 210, "t": 56, "b": 40}, legend={"orientation": "h", "y": -0.18})
fig.update_xaxes(type="category", nticks=10, gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
fig.update_yaxes(title_text="vol points", gridcolor=theme.grid, tickfont={"color": theme.text_secondary})
fig.show()

## 10. Check summary (bands fixed before looking at results)

In [11]:
ok = vix.dropna(subset=["vix"])
index_opts = ok["underlying"].isin(["OP000016.SH", "OP000300.SH", "OP000852.SH"])
ratio = ok.groupby("underlying")["vix"].mean() / ok.groupby("underlying")["realized_20d"].mean()
checks = pd.DataFrame([
    {"check": "0 coverage", "value": f"{len(ok)}/{len(vix)} underlying-days", "pass rule": "no failed rows", "pass": len(ok) == len(vix)},
    {"check": "1 synthetic flat 20%", "value": f"{synthetic_vix:.3f}", "pass rule": "|VIX - 20| < 0.25", "pass": abs(synthetic_vix - 20) < 0.25},
    {"check": "2 forward / spot, ETF options", "value": f"{ok.loc[~index_opts, 'f_over_spot'].min():.4f} .. {ok.loc[~index_opts, 'f_over_spot'].max():.4f}",
     "pass rule": "within 0.99 .. 1.01", "pass": bool(ok.loc[~index_opts, "f_over_spot"].between(0.99, 1.01).all())},
    {"check": "2b forward / spot, index options", "value": f"{ok.loc[index_opts, 'f_over_spot'].min():.4f} .. {ok.loc[index_opts, 'f_over_spot'].max():.4f}",
     "pass rule": "within 0.97 .. 1.01", "pass": bool(ok.loc[index_opts, "f_over_spot"].between(0.97, 1.01).all())},
    {"check": "3 VIX - ATM IV", "value": f"{ok['vix_minus_atm'].min():.2f} .. {ok['vix_minus_atm'].max():.2f}",
     "pass rule": "every row within -2 .. +5", "pass": bool(ok["vix_minus_atm"].between(-2, 5).all())},
    {"check": "4 VIX / realized (means)", "value": f"{ratio.min():.2f} .. {ratio.max():.2f}", "pass rule": "every underlying 0.5 .. 3",
     "pass": bool(ratio.between(0.5, 3).all())},
])
checks

,check,value,pass rule,pass
0,0 coverage,468/468 underlying-days,no failed rows,True
1,1 synthetic flat 20%,20.141,|VIX - 20| < 0.25,True
2,"2 forward / spot, ETF options",0.9887 .. 1.0010,within 0.99 .. 1.01,False
3,"2b forward / spot, index options",0.9835 .. 0.9998,within 0.97 .. 1.01,True
4,3 VIX - ATM IV,-2.02 .. 6.13,every row within -2 .. +5,False
5,4 VIX / realized (means),0.86 .. 1.19,every underlying 0.5 .. 3,True


## 11. Rows outside the bands

In [12]:
outside_forward = ok[~index_opts & ~ok["f_over_spot"].between(0.99, 1.01)]
outside_atm = ok[~ok["vix_minus_atm"].between(-2, 5)]
print(outside_forward.groupby("underlying").agg(rows=("trade_date", "size"), worst=("f_over_spot", "min")).round(4).to_string())
print()
print(outside_atm[["trade_date", "underlying", "vix", "atm_iv_30d", "vix_minus_atm", "n_options_near", "n_options_next"]].round(2).to_string(index=False))
catalog.close()

             rows   worst
underlying               
OP159901.SZ     2  0.9895
OP159922.SZ     1  0.9888
OP510500.SH     3  0.9887
OP588080.SH     1  0.9899

trade_date  underlying   vix  atm_iv_30d  vix_minus_atm  n_options_near  n_options_next
  20260917 OP159901.SZ 27.74       22.40           5.34            11.0             5.0
  20260917 OP159922.SZ 23.45       25.47          -2.02            10.0            17.0
  20260820 OP588000.SH 47.61       42.21           5.40            30.0            26.0
  20260820 OP588080.SH 48.12       41.99           6.13            28.0            25.0
  20260821 OP588080.SH 44.45       39.00           5.44            28.0            25.0
